In [ ]:
# ColabでTensorFlowをインストール（ランタイム起動直後に1回だけ実行）
!pip install tensorflow

In [ ]:
# 学習で使うライブラリを読み込み
import tensorflow as tf
import keras # TensorFlowの高レベルAPI
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.applications import EfficientNetB0

In [ ]:
# CIFAR-10データセットを読み込み（学習用50000とテスト用10000すでに分割済み）
(x_train, y_train), (x_test, y_test) = datasets.cifar10.load_data()

170498071/170498071 [==============================] - 13s 0us/step


In [ ]:
# 画像データの型を確認（通常はNumPy配列）、配列になっている
type(x_train)

numpy.ndarray

In [ ]:
# 学習データの配列サイズを確認（件数・画像サイズ・チャンネル数）
print(f'x_train shape:{x_train.shape}')
print(f'y_train shape:{y_train.shape}')

# x_train shape:(50000, 32, 32, 3) -> 50000枚の32x32 RGB画像
# y_train shape:(50000, 1) -> 50000件分のクラスラベル

x_train shape:(50000, 32, 32, 3)
y_train shape:(50000, 1)


In [ ]:
# テストデータの配列サイズも同様に確認
print(f'x_test shape:{x_test.shape}')
print(f'y_test shape:{y_test.shape}')

x_test shape:(10000, 32, 32, 3)
y_test shape:(10000, 1)


In [ ]:
# CIFAR-10は10クラス分類なのでクラス数を定義
num_classes = 10

# ラベルをワンホット表現に変換（例: 3 -> [0,0,0,1,0,0,0,0,0,0]）
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

In [ ]:
# ワンホット化後のラベル形状を確認（50000, 10 になる）
print(f'y_train shape:{y_train.shape}')

y_train shape:(50000, 10)


In [ ]:
# 変換後ラベルの中身を確認（0/1で各クラスを表現）該当するクラスに1が入る
y_train

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.],
       ...,
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.]], dtype=float32)

In [ ]:
# 画像データの型を確認（変換前はuint8）
x_train.dtype

dtype('uint8')

In [ ]:
# 学習時に扱いやすいようにfloat32へ変換
x_train = x_train.astype("float32")
x_test = x_test.astype("float32")

In [ ]:
# 変換後の型を再確認
x_train.dtype

dtype('float32')

In [ ]:
# 事前学習済みEfficientNetB0を読み込み（分類ヘッドは除外:理由　CIFAR-10は10クラス分類で、EfficientNetB0の分類ヘッドは1000クラス用なので、除外して新たに作成する必要があるため）
base_model = EfficientNetB0(input_shape=(32, 32, 3), include_top=False, weights="imagenet") # include_top=Falseで分類ヘッドを除外、weights="imagenet"でImageNetの重みを使用

16705208/16705208 [==============================] - 2s 0us/step


In [ ]:
# 転移学習用モデルを作成
# 1) 特徴抽出器(base_model)
# 2) 特徴マップを1次元化(GlobalAveragePooling2D)
# 3) 10クラス分類用の全結合層(softmax)
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(num_classes, activation="softmax")
])

In [ ]:
# モデルの学習設定
# optimizer: 重み更新アルゴリズム
# loss: 多クラス分類用の損失関数
# metrics: 学習中に表示する評価指標
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [ ]:
# モデル構造（層の並びとパラメータ数）を確認
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 efficientnetb0 (Functional  (None, 1, 1, 1280)        4049571   
 )                                                               
                                                                 
 global_average_pooling2d (  (None, 1280)              0         
 GlobalAveragePooling2D)                                         
                                                                 
 dense (Dense)               (None, 10)                12810     
                                                                 
Total params: 4062381 (15.50 MB)
Trainable params: 4020358 (15.34 MB)
Non-trainable params: 42023 (164.16 KB)
_________________________________________________________________


In [ ]:
# モデルを学習
# batch_size=32: 32枚ずつ学習
# epochs=3: 全データを3周
# validation_split=0.2: 学習データの20%を検証用に使用
history = model.fit(x_train, y_train, batch_size=32, epochs=3, validation_split=0.2)

Epoch 1/3
1250/1250 [==============================] - 99s 46ms/step - loss: 1.1067 - accuracy: 0.6281 - val_loss: 0.7233 - val_accuracy: 0.7546
Epoch 2/3
1250/1250 [==============================] - 55s 44ms/step - loss: 0.7432 - accuracy: 0.7532 - val_loss: 0.6173 - val_accuracy: 0.7872
Epoch 3/3
1250/1250 [==============================] - 54s 43ms/step - loss: 0.6132 - accuracy: 0.7943 - val_loss: 0.5598 - val_accuracy: 0.8106


In [ ]:
# テストデータで最終性能を評価
_, acc = model.evaluate(x_test, y_test)

313/313 [==============================] - 3s 10ms/step - loss: 0.5867 - accuracy: 0.8001


In [ ]:
# 精度を%表示で確認
print(f"accuracy:{acc * 100}%")

accuracy:80.0100028514862%


In [ ]:
# メモ: 追加実験（データ拡張・学習率調整・エポック増加）を試すセル